# 06 — LaMa Baseline Evaluation (Kaggle)

Chạy **LaMa** (Large Mask inpainting, WACV 2022) trên **1,998 ảnh test** (cùng split với `full-eval.ipynb`) để làm baseline so sánh.

**Pipeline:** `x_occ + mask → LaMa → x_hat`

**Metrics:** L1, L2, ICP, SS (Yan et al. 2019) + PSNR, SSIM, LPIPS, FID

**Test split:** 666 ảnh/bin × 3 bins (20-40%, 40-60%, 60-80%), seed=42 — **khớp với full-eval.ipynb**

In [ ]:
# CELL 1 — Dependencies
# Bước 1: pin numpy trước để tránh binary incompatibility với pandas
import sys
import subprocess
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', 'numpy==1.26.4',
], check=True)

# Bước 2: install các package còn lại
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'simple-lama-inpainting',
    'scikit-image', 'lpips', 'clean-fid',
    'opencv-python-headless', 'tqdm',
], check=True)

print('Dependencies ready')
print('>>> Sau khi cell này xong, hãy RESTART KERNEL rồi chạy lại từ Cell 2.')

In [ ]:
# CELL 2 — Config & Paths (Kaggle)
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Dataset (Kaggle) — CHỈNH tên dataset cho khớp với Kaggle notebook của bạn ──
DATASET_ROOT = Path("/kaggle/input/datasets/dangvy1507/vehicle")
SYNTH_DIR    = DATASET_ROOT / "synthetic_occ"
GT_DIR       = SYNTH_DIR / "x_gt"
OCC_DIR      = SYNTH_DIR / "x_occ"
MASK_DIR     = SYNTH_DIR / "masks"
META_CSV     = SYNTH_DIR / "metadata_synthetic_occ.csv"

# ── Output ────────────────────────────────────────────────────────────────────
OUT_BASE      = Path("/kaggle/working/eval_lama")
PRED_DIR      = OUT_BASE / "x_hat"
EVAL_GT_DIR   = OUT_BASE / "fid_gt"
EVAL_PRED_DIR = OUT_BASE / "fid_pred"
REPORT_DIR    = OUT_BASE / "reports"
for d in [PRED_DIR, EVAL_GT_DIR, EVAL_PRED_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Bins (khớp với full-eval.ipynb) ───────────────────────────────────────────
BIN_EDGES  = [(0.20, 0.40), (0.40, 0.60), (0.60, 0.80)]
BIN_LABELS = ["20-40%", "40-60%", "60-80%"]
TEST_SIZE  = 2000      # 666/bin x 3 = 1,998 thực tế

assert SYNTH_DIR.exists(), f"Dataset không tìm thấy: {SYNTH_DIR}"
assert META_CSV.exists(),  f"Metadata không tìm thấy: {META_CSV}"

meta = pd.read_csv(META_CSV)
meta.columns = meta.columns.str.strip().str.lower()
print(f"Dataset    : {len(meta):,} images")
print(f"Device     : {DEVICE}")
print(f"Test target: {TEST_SIZE} (~ 666/bin x 3 bins)")

In [ ]:
# CELL 3 — Tạo Test Set (666/bin x 3 bins, seed=42 — GIỐNG HỆT full-eval.ipynb)
_per_bin = TEST_SIZE // len(BIN_EDGES)   # = 666
_bin_dfs = []
for lo, hi in BIN_EDGES:
    sub = meta[(meta['occlusion_ratio'] >= lo) & (meta['occlusion_ratio'] < hi)]
    _bin_dfs.append(sub.sample(min(_per_bin, len(sub)), random_state=SEED))
test_df = pd.concat(_bin_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Gán bin_label
def assign_bin(r):
    for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
        if lo <= r < hi:
            return lbl
    return '?'
test_df['bin_label'] = test_df['occlusion_ratio'].apply(assign_bin)

print(f"Test set: {len(test_df):,} images")
for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
    n = len(test_df[(test_df['occlusion_ratio'] >= lo) & (test_df['occlusion_ratio'] < hi)])
    print(f"  {lbl}: {n} images")

In [ ]:
# CELL 4 — Load LaMa Model
from simple_lama_inpainting import SimpleLama

print("Loading LaMa model (auto-download pretrained weights ~200 MB) ...")
lama = SimpleLama()
print(f"LaMa loaded | device={DEVICE}")

In [ ]:
# CELL 5 — Preview 4 mẫu trước khi inference
import cv2
from PIL import Image
import matplotlib.pyplot as plt

samples_vis = test_df.sample(4, random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for i, row in samples_vis.iterrows():
    occ  = Image.open(OCC_DIR  / row['x_occ']).convert('RGB').resize((512, 512))
    msk  = Image.open(MASK_DIR / row['mask']).convert('L').resize((512, 512))
    gt   = Image.open(GT_DIR   / row['x_gt']).convert('RGB').resize((512, 512))
    pred = lama(occ, msk)   # preview nhanh

    for j, (img, title) in enumerate([(occ,'x_occ'), (msk,'Mask'), (pred,'LaMa pred'), (gt,'x_gt (GT)')]):
        ax = axes[i, j]
        ax.imshow(img, cmap='gray' if title == 'Mask' else None)
        if i == 0: ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')
    axes[i, 0].set_ylabel(f"occ={row['occlusion_ratio']:.2f}", fontsize=8)

plt.suptitle('Preview — LaMa inpainting', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6 — Full Inference trên 1,998 ảnh test
import time
from tqdm.auto import tqdm

print(f"Inference: {len(test_df):,} images (LaMa)")
pred_records = []
t0 = time.time()

for _, r in tqdm(test_df.iterrows(), total=len(test_df), desc='LaMa inference'):
    occ  = Image.open(OCC_DIR  / r['x_occ']).convert('RGB').resize((512, 512))
    msk  = Image.open(MASK_DIR / r['mask']).convert('L').resize((512, 512))
    gt   = Image.open(GT_DIR   / r['x_gt']).convert('RGB').resize((512, 512))

    pred = lama(occ, msk)          # SimpleLama: mask=255 vùng cần fill
    pred = pred.resize((512, 512)) # đảm bảo đúng kích thước

    fname = f"{r['stem']}.png"
    pred.save(PRED_DIR      / fname)
    gt.save  (EVAL_GT_DIR   / fname)
    pred.save(EVAL_PRED_DIR / fname)

    pred_records.append({
        'stem': r['stem'],
        'x_gt': r['x_gt'], 'x_occ': r['x_occ'], 'mask': r['mask'],
        'occlusion_ratio': r['occlusion_ratio'],
        'bin_label': r['bin_label'],
        'pred': fname,
    })

elapsed  = time.time() - t0
pred_df  = pd.DataFrame(pred_records)
pred_df.to_csv(REPORT_DIR / 'pred_index.csv', index=False)
print(f"Done: {len(pred_df):,} images | {elapsed/60:.1f} min | ~{elapsed/max(len(pred_df),1):.2f}s/img")

In [ ]:
# CELL 7 — Định nghĩa Metric Functions
import lpips as lpips_lib
from skimage.metrics import (
    structural_similarity as ssim_fn,
    peak_signal_noise_ratio as psnr_fn,
)

lpips_model = lpips_lib.LPIPS(net='alex').to(DEVICE).eval()

def pixel_l1(gt, pred, mask01):
    gt_f = gt.astype(np.float64) / 255.
    pr_f = pred.astype(np.float64) / 255.
    m = mask01.astype(bool)
    return float(np.mean(np.abs(gt_f[m] - pr_f[m]))) if m.sum() > 0 \
           else float(np.mean(np.abs(gt_f - pr_f)))

def pixel_l2(gt, pred, mask01):
    gt_f = gt.astype(np.float64) / 255.
    pr_f = pred.astype(np.float64) / 255.
    m = mask01.astype(bool)
    return float(np.mean((gt_f[m] - pr_f[m]) ** 2)) if m.sum() > 0 \
           else float(np.mean((gt_f - pr_f) ** 2))

def masked_psnr(gt, pred, mask01):
    m = mask01.astype(bool)
    gt_m = gt[m].astype(np.float64)
    pr_m = pred[m].astype(np.float64)
    mse = np.mean((gt_m - pr_m) ** 2) if m.sum() > 0 \
          else np.mean((gt.astype(np.float64) - pred.astype(np.float64)) ** 2)
    return float(10 * np.log10(255. ** 2 / mse)) if mse > 0 else 100.0

def masked_ssim(gt, pred, mask01):
    _, smap = ssim_fn(
        gt.astype(np.float32) / 255.,
        pred.astype(np.float32) / 255.,
        channel_axis=2, data_range=1.0, full=True)
    m = mask01.astype(bool)
    return float(smap[m].mean()) if m.sum() > 0 else float(smap.mean())

def masked_lpips(gt, pred, mask01):
    m = mask01.astype(np.float32)[..., None]
    gt_t = torch.from_numpy((gt * m).astype(np.uint8)).permute(2,0,1).unsqueeze(0).float() / 127.5 - 1.
    pr_t = torch.from_numpy((pred * m).astype(np.uint8)).permute(2,0,1).unsqueeze(0).float() / 127.5 - 1.
    with torch.no_grad():
        return float(lpips_model(gt_t.to(DEVICE), pr_t.to(DEVICE)).item())

print('Metric functions defined: pixel_l1 / pixel_l2 / masked_psnr / masked_ssim / masked_lpips')

In [ ]:
# CELL 8 — Load Inception V3 (ICP) + DeepLab V3 (SS)
from torchvision import transforms
from torchvision.models import inception_v3
from torchvision.models.segmentation import deeplabv3_resnet101

# ── Inception V3 (ICP) ────────────────────────────────────────────────────────
print('Loading Inception V3 ...')
try:
    from torchvision.models import Inception_V3_Weights
    inception_net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
except (ImportError, AttributeError):
    inception_net = inception_v3(pretrained=True)
inception_net.eval().to(DEVICE)

IMAGENET_CAR_CLASSES = [407, 436, 511, 627, 656, 705, 717, 734, 751, 779, 817, 820, 868]
inception_tf = transforms.Compose([
    transforms.Resize(299), transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def compute_icp(pred_pil, mask_np_gray):
    ys, xs = np.where(mask_np_gray > 127)
    if len(ys) == 0:
        return 0.0
    pad = 16; h, w = mask_np_gray.shape
    y0 = max(0, int(ys.min()) - pad); y1 = min(h, int(ys.max()) + pad)
    x0 = max(0, int(xs.min()) - pad); x1 = min(w, int(xs.max()) + pad)
    crop = pred_pil.crop((x0, y0, x1, y1))
    if crop.width < 10 or crop.height < 10:
        return 0.0
    inp = inception_tf(crop.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out    = inception_net(inp)
        logits = out.logits if hasattr(out, 'logits') else out
        probs  = torch.softmax(logits, dim=1)[0].cpu()
    return float(probs[IMAGENET_CAR_CLASSES].sum())

# ── DeepLab V3 (SS) ───────────────────────────────────────────────────────────
print('Loading DeepLab V3 ...')
try:
    from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights
    deeplab_net = deeplabv3_resnet101(weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1)
except (ImportError, AttributeError):
    deeplab_net = deeplabv3_resnet101(pretrained=True)
deeplab_net.eval().to(DEVICE)

CAR_CLASS_IDX = 7
seg_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def compute_ss(pred_pil, mask01):
    inp = seg_tf(pred_pil.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        seg_out = deeplab_net(inp)['out'][0]
    pred_seg = seg_out.argmax(0).cpu().numpy().astype(np.uint8)
    pred_seg = cv2.resize(pred_seg, (512, 512), interpolation=cv2.INTER_NEAREST)
    m = mask01.astype(bool)
    return float((pred_seg[m] == CAR_CLASS_IDX).mean()) if m.sum() > 0 else 0.0

print('Models ready: Inception V3 (ICP) + DeepLab V3 (SS)')

In [ ]:
# CELL 9 — Compute Per-image Metrics (L1, L2, ICP, SS, PSNR, SSIM, LPIPS)
from tqdm.auto import tqdm

print(f'Computing metrics on {len(pred_df):,} images ...')
metric_rows = []

for _, r in tqdm(pred_df.iterrows(), total=len(pred_df), desc='Metrics'):
    gt_bgr = cv2.imread(str(GT_DIR   / r['x_gt']))
    pr_bgr = cv2.imread(str(PRED_DIR / r['pred']))
    if gt_bgr is None or pr_bgr is None:
        continue
    gt_np  = cv2.resize(cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    pr_np  = cv2.resize(cv2.cvtColor(pr_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    mk_raw = cv2.imread(str(MASK_DIR / r['mask']), cv2.IMREAD_GRAYSCALE)
    mk_np  = cv2.resize(mk_raw, (512, 512))
    mk01   = (mk_np > 127).astype(np.uint8)
    pr_pil = Image.fromarray(pr_np)

    metric_rows.append({
        'stem'            : r['stem'],
        'occlusion_ratio' : r['occlusion_ratio'],
        'bin_label'       : r['bin_label'],
        'l1'   : pixel_l1(gt_np, pr_np, mk01),
        'l2'   : pixel_l2(gt_np, pr_np, mk01),
        'icp'  : compute_icp(pr_pil, mk_np),
        'ss'   : compute_ss(pr_pil, mk01),
        'psnr' : masked_psnr(gt_np, pr_np, mk01),
        'ssim' : masked_ssim(gt_np, pr_np, mk01),
        'lpips': masked_lpips(gt_np, pr_np, mk01),
    })

metric_df = pd.DataFrame(metric_rows)
metric_df.to_csv(REPORT_DIR / 'metrics_per_image.csv', index=False)
print(f'\nOverall ({len(metric_df):,} images):')
print(metric_df[['l1','l2','icp','ss','psnr','ssim','lpips']].describe().round(4))

In [ ]:
# CELL 10 — FID
from cleanfid import fid as cleanfid

print('Computing FID (clean-fid) ...')
fid_value = cleanfid.compute_fid(
    str(EVAL_GT_DIR), str(EVAL_PRED_DIR), mode='clean')
print(f'FID = {fid_value:.2f}')

In [ ]:
# CELL 11 — Report Table: Overall + Per-bin + So sánh với full-eval + Yan et al.
ALL_METRICS = ['l1', 'l2', 'icp', 'ss', 'psnr', 'ssim', 'lpips']

def bin_summary(df, lo, hi, label):
    sub = df[(df['occlusion_ratio'] >= lo) & (df['occlusion_ratio'] < hi)]
    if len(sub) == 0:
        return {'bin': label, 'n': 0, **{m: None for m in ALL_METRICS}}
    row = {'bin': label, 'n': len(sub)}
    for m in ALL_METRICS:
        row[m] = round(float(sub[m].mean()), 4)
    return row

overall = {'bin': 'overall', 'n': len(metric_df)}
for m in ALL_METRICS:
    overall[m] = round(float(metric_df[m].mean()), 4)

rows = [overall]
for (lo, hi), lbl in zip(BIN_EDGES, BIN_LABELS):
    rows.append(bin_summary(metric_df, lo, hi, lbl))

summary_df = pd.DataFrame(rows)
summary_df.insert(
    summary_df.columns.get_loc('psnr') + 3, 'fid',
    [round(fid_value, 2)] + [None] * len(BIN_EDGES))

print('=' * 80)
print('RESULTS — LaMa Baseline (pretrained on Places2)')
print(f'n={len(metric_df):,}  |  FID={fid_value:.2f}')
print('=' * 80)
print(summary_df.to_string(index=False))
print('=' * 80)

# ── So sánh 3 cấu hình trên cùng test set ─────────────────────────────────────
# Kết quả từ full-eval.ipynb (SD1.5 + CN + IPA, không LoRA) — hardcode từ notebook đó
sdipa_vals = {
    'l1': 0.1438, 'l2': 0.0521, 'icp': 0.6081, 'ss': 0.3319,
    'psnr': 13.5475, 'ssim': 0.4151, 'lpips': 0.1285, 'fid': None,
}
lama_vals = {m: overall.get(m) for m in ALL_METRICS}
lama_vals['fid'] = round(fid_value, 2)

print('\n' + '=' * 80)
print('SO SÁNH: LaMa (baseline) vs SD1.5+CN+IPA (full-eval.ipynb)')
print('=' * 80)
cmp_rows = [
    {'method': 'LaMa (Places2 pretrained)',  **{m: lama_vals[m]  for m in ALL_METRICS}, 'fid': lama_vals['fid']},
    {'method': 'SD1.5 + ControlNet + IPA',  **{m: sdipa_vals[m] for m in ALL_METRICS}, 'fid': sdipa_vals['fid']},
]
cmp_df = pd.DataFrame(cmp_rows)
pd.set_option('display.float_format', '{:.4f}'.format)
print(cmp_df.to_string(index=False))
print('(L1↓ L2↓ ICP↑ SS↑ PSNR↑ SSIM↑ LPIPS↓ FID↓)')

# ── So sánh với Yan et al. (ICCV 2019) ────────────────────────────────────────
print('\n' + '=' * 80)
print('SO SÁNH VỚI Yan et al. (ICCV 2019) — Synthetic M^gt')
print('=' * 80)
paper_rows = [
    {'method': 'Deepfill [50]',      'l1': 0.0284, 'l2': 0.0107, 'icp': 0.5620, 'ss': 0.8295},
    {'method': 'Liu et al. [27]',    'l1': 0.0272, 'l2': 0.0074, 'icp': 0.6284, 'ss': 0.8672},
    {'method': 'Pathak et al. [35]', 'l1': 0.0207, 'l2': 0.0088, 'icp': 0.5708, 'ss': 0.8517},
    {'method': 'pix2pix [20]',       'l1': 0.0174, 'l2': 0.0060, 'icp': 0.7081, 'ss': 0.9410},
    {'method': 'SeGAN [12]',         'l1': 0.0181, 'l2': 0.0055, 'icp': 0.6662, 'ss': 0.9371},
    {'method': 'Yan et al. (best)',  'l1': 0.0158, 'l2': 0.0038, 'icp': 0.7436, 'ss': 0.9458},
    {'method': '--- LaMa (ours baseline) ---',
     'l1': lama_vals['l1'], 'l2': lama_vals['l2'],
     'icp': lama_vals['icp'], 'ss': lama_vals['ss']},
    {'method': '--- SD1.5+CN+IPA (full-eval) ---',
     'l1': sdipa_vals['l1'], 'l2': sdipa_vals['l2'],
     'icp': sdipa_vals['icp'], 'ss': sdipa_vals['ss']},
]
paper_df = pd.DataFrame(paper_rows)
print(paper_df.to_string(index=False))
print('=' * 80)
print('Lưu ý: Yan et al. dùng dataset khác (không phải Stanford Cars + COCO synthetic).')
print('So sánh L1/L2/ICP/SS mang tính tham chiếu, không trực tiếp.')

# ── Save ──────────────────────────────────────────────────────────────────────
summary_df.to_csv(REPORT_DIR / 'lama_summary.csv', index=False)
cmp_df.to_csv(REPORT_DIR / 'comparison_lama_vs_sdipa.csv', index=False)
paper_df.to_csv(REPORT_DIR / 'paper_comparison.csv', index=False)

import json
with open(REPORT_DIR / 'eval_config.json', 'w') as f:
    json.dump({
        'method': 'LaMa (simple-lama-inpainting)',
        'n': int(len(metric_df)),
        **{m: overall.get(m) for m in ALL_METRICS},
        'fid': round(fid_value, 2),
    }, f, indent=2)

print('\nFiles saved:', [f.name for f in sorted(REPORT_DIR.glob('*'))])

In [ ]:
# CELL 12 — Visualization
import matplotlib.pyplot as plt

# 12a. Bar charts: metrics theo bin
plot_df = summary_df[summary_df['bin'] != 'overall'].copy()
metrics_plot = [
    ('psnr', 'PSNR ↑ (dB)', True),
    ('ssim', 'SSIM ↑',      True),
    ('lpips','LPIPS ↓',     False),
    ('l1',   'L1 ↓',        False),
]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
colors = ['#3498db', '#2ecc71', '#e74c3c']
for ax, (col, label, _) in zip(axes, metrics_plot):
    bars = ax.bar(plot_df['bin'], plot_df[col], color=colors, edgecolor='white', width=0.5)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Occlusion level')
    for bar, v in zip(bars, plot_df[col]):
        if v is not None:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.grid(alpha=0.3)
plt.suptitle('LaMa — Metrics by Occlusion Level', y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'metrics_by_bin.png', dpi=150, bbox_inches='tight')
plt.show()

# 12b. PSNR scatter
fig2, ax2 = plt.subplots(figsize=(8, 4))
sc = ax2.scatter(metric_df['occlusion_ratio'], metric_df['psnr'],
                 c=metric_df['psnr'], cmap='RdYlGn', alpha=0.4, s=8)
plt.colorbar(sc, ax=ax2, label='PSNR (dB)')
for (lo, hi) in BIN_EDGES:
    ax2.axvline(lo, color='gray', linestyle='--', linewidth=0.8)
ax2.axvline(BIN_EDGES[-1][1], color='gray', linestyle='--', linewidth=0.8)
ax2.set_xlabel('Occlusion ratio')
ax2.set_ylabel('PSNR (dB)')
ax2.set_title('PSNR vs Occlusion Ratio — LaMa')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'psnr_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# 12c. Visual grid: best + worst theo PSNR mỗi bin
fig4, axes4 = plt.subplots(len(BIN_EDGES) * 2, 4, figsize=(14, 5 * len(BIN_EDGES) * 2))
row_idx = 0
for (lo, hi), label in zip(BIN_EDGES, BIN_LABELS):
    bin_m = metric_df[(metric_df['occlusion_ratio'] >= lo) & (metric_df['occlusion_ratio'] < hi)]
    if len(bin_m) == 0:
        continue
    best_stem  = bin_m.loc[bin_m['psnr'].idxmax(), 'stem']
    worst_stem = bin_m.loc[bin_m['psnr'].idxmin(), 'stem']
    for stem, tag in [(best_stem, 'BEST'), (worst_stem, 'WORST')]:
        row_r = pred_df[pred_df['stem'] == stem].iloc[0]
        m_row = bin_m[bin_m['stem'] == stem].iloc[0]
        imgs = [
            (Image.open(GT_DIR   / row_r['x_gt']).convert('RGB'),  'x_gt'),
            (Image.open(OCC_DIR  / row_r['x_occ']).convert('RGB'), 'x_occ'),
            (Image.open(MASK_DIR / row_r['mask']).convert('L'),    'Mask'),
            (Image.open(PRED_DIR / row_r['pred']).convert('RGB'),  'LaMa pred'),
        ]
        for col_j, (img, title) in enumerate(imgs):
            ax = axes4[row_idx, col_j]
            ax.imshow(img, cmap='gray' if title == 'Mask' else None)
            if row_idx == 0:
                ax.set_title(title, fontsize=9, fontweight='bold')
            ax.axis('off')
        axes4[row_idx, 0].set_ylabel(
            f'Bin {label}\n{tag}\nPSNR={m_row["psnr"]:.1f}  SSIM={m_row["ssim"]:.3f}\nLPIPS={m_row["lpips"]:.3f}',
            fontsize=7, labelpad=4)
        row_idx += 1

for ax in axes4[row_idx:].flatten():
    ax.axis('off')
plt.suptitle('Best & Worst per Bin — LaMa', y=1.01)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'visual_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 13 — Zip kết quả
import zipfile

zip_path = OUT_BASE / 'lama_eval_reports.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=3) as zf:
    for f in sorted(REPORT_DIR.glob('*')):
        zf.write(f, f'reports/{f.name}')

print(f'Zip: {zip_path.name}  ({zip_path.stat().st_size / 1e6:.1f} MB)')
print('Contains:', [f.name for f in sorted(REPORT_DIR.glob('*'))])
print(f'\n>>> Download: /kaggle/working/eval_lama/lama_eval_reports.zip')